# Chapter 4: Vermilion City — Matching & Subclassification*The S.S. Anne is docked at Vermilion Harbor. Among 400 passengers, some completed Lt. Surge's Thunder Training. Your mission: find each treated trainer's causal "twin" to estimate the true effect on the Vermilion Gym battle.*## Learning ObjectivesBy the end of this notebook you will be able to:- Apply exact, coarsened, Mahalanobis, and propensity-score matching on a real dataset- Interpret covariate balance tables and Love plots- Estimate the ATT via nearest-neighbor matching and subclassification- Recognize when matching on observables is insufficient (hidden confounders)

In [ ]:
import sys, ossys.path.insert(0, os.path.abspath('../src'))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy.spatial.distance import mahalanobisfrom scipy.stats import zscorefrom kanto_utils import (    load_ss_anne, apply_kanto_theme, love_plot, balance_table,    propensity_score, oak_says, blue_says, blues_mistake, badge_earned)apply_kanto_theme()np.random.seed(151)oak_says("Welcome aboard the S.S. Anne! Let's find causal twins among 400 passengers.")

## 1. Load the DataThe dataset contains 400 passengers. `thunder_training` is the treatment, `vermilion_gym_win` is the outcome. The dataset includes two *hidden* confounders — `patience` and `natural_talent` — which we will pretend not to see at first.

In [ ]:
df = load_ss_anne()print(f"Shape: {df.shape}")print(f"Treated: {df.thunder_training.sum()} | Control: {(1 - df.thunder_training).sum()}")df.head()

## 2. Naive ComparisonLet's start with the lazy approach: just compare treated to control directly.

In [ ]:
naive_att = df.loc[df.thunder_training == 1, 'vermilion_gym_win'].mean() - \            df.loc[df.thunder_training == 0, 'vermilion_gym_win'].mean()print(f"Naive difference in means: {naive_att:.3f}")blues_mistake(    "Thunder Training works! Graduates win 30% more often than non-graduates!",    "That 30% gap reflects who CHOSE Thunder Training — more experienced, higher-wealth trainers with better teams — not the training itself.")

## 3. Covariate ImbalanceBefore matching, the treated and control groups look very different on every covariate.

In [ ]:
observed_covs = ['badges', 'team_level_avg', 'trainer_experience', 'play_hours',                 'team_size', 'strategy_score', 'wealth', 'age']bal_before = balance_table(df, 'thunder_training', observed_covs)bal_before

## 4. Mahalanobis Distance Matching (1:1 Nearest Neighbor)For each treated trainer, find the nearest untreated trainer in Mahalanobis distance and pair them.

In [ ]:
X = df[observed_covs].valuescov_inv = np.linalg.pinv(np.cov(X, rowvar=False))treated_idx = np.where(df.thunder_training.values == 1)[0]control_idx = np.where(df.thunder_training.values == 0)[0]matches = []used = set()for t in treated_idx:    best_dist, best_c = np.inf, None    for c in control_idx:        if c in used:            continue        diff = X[t] - X[c]        d = np.sqrt(diff @ cov_inv @ diff)        if d < best_dist:            best_dist, best_c = d, c    matches.append((t, best_c, best_dist))    used.add(best_c)matched_df = pd.concat([    df.iloc[[t for t,_,_ in matches]].assign(pair=range(len(matches)), role='treated'),    df.iloc[[c for _,c,_ in matches]].assign(pair=range(len(matches)), role='control'),])att_mahal = (df.iloc[[t for t,_,_ in matches]].vermilion_gym_win.mean() -             df.iloc[[c for _,c,_ in matches]].vermilion_gym_win.mean())print(f"ATT (Mahalanobis 1:1 matching): {att_mahal:.3f}")

## 5. Propensity Score EstimationFit a logistic regression for treatment assignment, then use the estimated scores to match and subclassify.

In [ ]:
ps_result = propensity_score(X, df.thunder_training.values)df['propensity'] = ps_result['propensity_scores']fig, ax = plt.subplots(figsize=(9, 5))ax.hist(df.loc[df.thunder_training == 1, 'propensity'], bins=30, alpha=0.6, label='Treated', color='#EE1515')ax.hist(df.loc[df.thunder_training == 0, 'propensity'], bins=30, alpha=0.6, label='Control', color='#3B4CCA')ax.set_xlabel('Estimated propensity score')ax.set_ylabel('Count')ax.set_title('Propensity Score Overlap — S.S. Anne')ax.legend()plt.tight_layout()plt.show()

## 6. Propensity Score MatchingMatch each treated unit to the nearest control on propensity score alone.

In [ ]:
def ps_match(df, ps_col='propensity', treat_col='thunder_training'):    treated = df[df[treat_col] == 1].reset_index()    control = df[df[treat_col] == 0].reset_index()    used = set()    pairs = []    for _, t in treated.iterrows():        available = control[~control.index.isin(used)]        if len(available) == 0:            break        best = (available[ps_col] - t[ps_col]).abs().idxmin()        used.add(best)        pairs.append((t['index'], control.loc[best, 'index']))    return pairspairs = ps_match(df)treated_ids = [a for a, _ in pairs]control_ids = [b for _, b in pairs]att_ps = df.loc[treated_ids, 'vermilion_gym_win'].mean() - df.loc[control_ids, 'vermilion_gym_win'].mean()print(f"ATT (propensity score matching): {att_ps:.3f}")print(f"ATT (Mahalanobis matching):       {att_mahal:.3f}")print(f"Naive (no matching):              {naive_att:.3f}")

## 7. Subclassification by Propensity ScoreDivide into 5 strata based on propensity score quintiles and compute within-stratum treatment effects.

In [ ]:
df['ps_stratum'] = pd.qcut(df.propensity, 5, labels=False, duplicates='drop')strata_results = []for s in sorted(df.ps_stratum.unique()):    sub = df[df.ps_stratum == s]    n_t = (sub.thunder_training == 1).sum()    n_c = (sub.thunder_training == 0).sum()    if n_t == 0 or n_c == 0:        continue    effect = sub[sub.thunder_training == 1].vermilion_gym_win.mean() - \             sub[sub.thunder_training == 0].vermilion_gym_win.mean()    strata_results.append({'stratum': s, 'n_treated': n_t, 'n_control': n_c, 'effect': effect})strata_df = pd.DataFrame(strata_results)overall = np.average(strata_df.effect, weights=strata_df.n_treated + strata_df.n_control)print(f"Overall subclassification ATT: {overall:.3f}")strata_df

## 8. Love Plot: Balance ImprovementA Love plot shows standardized mean differences before and after matching. |SMD| < 0.1 is considered good balance.

In [ ]:
# Post-matching balancematched_ids = treated_ids + control_idsmatched = df.loc[matched_ids]bal_after = balance_table(matched, 'thunder_training', observed_covs)smd_before = bal_before['smd'].valuessmd_after  = bal_after['smd'].valuesfig, ax = plt.subplots(figsize=(9, 6))love_plot(smd_before, smd_after, observed_covs, ax=ax)ax.set_title("Love Plot: Before vs After Propensity-Score Matching")plt.tight_layout()plt.show()

## 9. The Hidden Confounders RevealMatching on observables only balances *observable* covariates. The `ss_anne_passengers` dataset has two hidden confounders — `patience` and `natural_talent` — that we ignored until now.Let's reveal them and check whether they are still imbalanced after matching.

In [ ]:
hidden = ['patience', 'natural_talent']bal_hidden = balance_table(matched, 'thunder_training', hidden)print("Hidden confounder balance AFTER propensity-score matching:")print(bal_hidden)oak_says('''Even after careful matching, the hidden confounders remain imbalanced.This is the fundamental limitation of selection-on-observables methods:you can only balance what you can measure. In Chapter 6, we'll meetinstrumental variables — a method that can handle unobserved confounders.''')

## Challenge Exercises**Challenge 1:** Implement k:1 matching (k=3) and compare to 1:1.**Challenge 2:** Add interaction and polynomial terms to the propensity score model. Does covariate balance improve?**Challenge 3:** Implement caliper matching with caliper=0.05 on the propensity score. How many treated units are dropped?

In [ ]:
badge_earned("Thunder Badge", 4)